# Pipeline de anonimizacao da base de vendas (Google Colab)

Gera, a partir da extracao real de vendas, um arquivo com **a mesma estrutura**
(mesmas colunas, mesmos tipos, mesma quantidade de linhas), porem com os
campos que identificam a empresa **trocados por codigos sinteticos**:

- SKU do produto (`Prod_SKU`) e o `Prod_Id` associado a ele;
- identificador de campanha (`Ped_Campanha`);
- categoria do produto (`Cat_CategoriaNome`, com nomes de marca/fornecedor
  reais) e o `Cat_CategoriaId` associado a ela.

## Finalidade

Os dados de vendas sao cobertos por acordo de confidencialidade com a empresa
e nao podem ser publicados como estao, porque SKU, campanha e categoria (nomes
de marca) permitem reidentificar produtos, marcas parceiras e acoes comerciais
da empresa. Este notebook produz a versao que acompanha o codigo no
repositorio publico (Secao 4.4.1).

Este notebook roda **antes** do pipeline principal
(`TCC_Pipeline_Colab.ipynb`) e produz:

1. **Arquivo anonimizado** (`*_ANONIMIZADO.csv`) — mesma estrutura da base
   original, distribuido no repositorio como `data/base_venda.csv`.
2. **Tabela De/Para** (`de_para_<coluna>.csv`) — o mapeamento
   valor-original -> codigo-sintetico. **Fica apenas em pasta privada e nunca
   deve ser versionada nem publicada**: e ela que permite reverter a
   anonimizacao.

O pipeline principal roda igual com qualquer um dos dois arquivos — a unica
coisa que muda e o parametro `DADOS` da celula de parametros dele.
`Cat_CategoriaNome` continua presente e utilizavel como feature
(`categoria_enc`) depois de anonimizada — so o nome real da marca deixa de
aparecer. Nenhuma outra coluna (datas, quantidades) e alterada, entao os
resultados de EDA e modelagem sao os mesmos com a base original ou com a
anonimizada; a diferenca e so o rotulo do SKU/campanha/categoria.

## 1. Montar o Google Drive

Funciona tambem fora do Colab (por exemplo, para testes locais): se o modulo
`google.colab` nao existir, o notebook so avisa e segue usando os caminhos
locais informados na celula de parametros.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    EM_COLAB = True
except ImportError:
    print('google.colab nao encontrado — rodando fora do Colab, usando caminhos locais.')
    EM_COLAB = False


## 2. Parametros (edite aqui)

Unica celula que precisa ser alterada no dia a dia.

In [ ]:
"""Parâmetros da execução."""
from pathlib import Path

# =============================================================================
# CAMINHOS
# =============================================================================

# CSV da base original (a mesma que alimenta DADOS no pipeline principal).
# Ajuste para o caminho da base antes de rodar.
CSV_ENTRADA = "/content/drive/MyDrive/dados/base_vendas.csv"

# Pasta onde o arquivo anonimizado (esse pode ir ao repositório publico) e
# salvo. Nome de saída = nome de entrada + sufixo ANONIMIZADO.
PASTA_SAIDA_ANONIMIZADA = "/content/drive/MyDrive/dados/anonimizado"

# Pasta da tabela De/Para. NUNCA aponte isso para uma pasta versionada no Git
# nem para o repositório publico — e o que permite desfazer a anonimização.
PASTA_DE_PARA_PRIVADA = "/content/drive/MyDrive/dados/privado/de_para"

# =============================================================================
# O QUE ANONIMIZAR
# =============================================================================

# Colunas a trocar por código sintético, e o prefixo do código gerado para
# cada uma. Adicionar uma coluna aqui e suficiente para passar a anonimiza-la
# também — não precisa mexer no restante do notebook.
#
# Os pares (Prod_SKU, Prod_Id) e (Cat_CategoriaNome, Cat_CategoriaId) entram
# juntos por padrão: em cada par, as duas colunas tem correspondência 1 para 1
# na base (um SKU tem exatamente um Prod_Id; uma categoria tem exatamente um
# Cat_CategoriaId). Publicar so um lado do par com o código trocado e deixar o
# outro no valor real anularia a troca — bastaria cruzar os dois para
# reidentificar o produto ou a marca/categoria. Cat_CategoriaId não é usado
# em nenhum cálculo do pipeline (so o Cat_CategoriaNome vira feature), mas
# recebe o mesmo tratamento pelo motivo acima.
COLUNAS_ANONIMIZAR = {
    "Prod_SKU": "SKU",
    "Prod_Id": None,             # None = numérico sequencial, ver _gerar_codigos abaixo
    "Ped_Campanha": "CAMP",
    "Cat_CategoriaNome": "CATEGORIA",
    "Cat_CategoriaId": None,     # None = numérico sequencial
}

# Semente para embaralhar a ordem de atribuição dos códigos sintéticos, para
# que a sequência do código NÃO reproduza nenhuma ordem (alfabética, temporal
# etc.) presente no valor original.
SEED = 42

# Se True, reaproveita o De/Para ja salvo em PASTA_DE_PARA_PRIVADA (so
# acrescenta código novo para valor que ainda não apareceu). Mantém o mesmo
# SKU/categoria sempre com o mesmo código entre execuções/bases diferentes.
# Se False, gera um mapeamento novo do zero a cada execução.
REUTILIZAR_MAPEAMENTO_EXISTENTE = True


## 3. Funcoes de anonimizacao

In [ ]:
import pandas as pd
import random
from pathlib import Path


def _gerar_codigos(valores_unicos, prefixo, seed):
    """Gera um código sintético por valor único, em ordem embaralhada.

    prefixo=None -> código numérico sequencial (mantém a coluna como inteiro).
    prefixo=str  -> código "<PREFIXO><índice com zero a esquerda>".
    """
    valores = list(valores_unicos)
    rng = random.Random(seed)
    rng.shuffle(valores)
    n_digitos = max(6, len(str(len(valores))))
    if prefixo is None:
        base = 900_000_001
        return {v: base + i for i, v in enumerate(valores)}
    return {v: f"{prefixo}{str(i + 1).zfill(n_digitos)}" for i, v in enumerate(valores)}


def carregar_ou_criar_mapeamento(coluna, valores_unicos, prefixo, seed,
                                  pasta_de_para, reutilizar):
    """Le o De/Para existente (se houver e reutilizar=True) e cria código novo
    somente para valores que ainda não tem correspondente."""
    pasta_de_para = Path(pasta_de_para)
    pasta_de_para.mkdir(parents=True, exist_ok=True)
    caminho = pasta_de_para / f"de_para_{coluna}.csv"

    mapeamento = {}
    if reutilizar and caminho.exists():
        de_para_existente = pd.read_csv(caminho, dtype=str)
        mapeamento = dict(zip(de_para_existente["original"].astype(str),
                               de_para_existente["anonimizado"].astype(str)))

    faltantes = [v for v in valores_unicos if str(v) not in mapeamento]
    if faltantes:
        novo = _gerar_codigos(faltantes, prefixo, seed)
        mapeamento.update({str(k): str(v) for k, v in novo.items()})

    pd.DataFrame({"original": list(mapeamento.keys()),
                  "anonimizado": list(mapeamento.values())}
                 ).to_csv(caminho, index=False)
    return mapeamento, caminho


def aplicar_anonimizacao(df, colunas_anonimizar, seed, pasta_de_para, reutilizar):
    """Aplica o De/Para a cada coluna configurada e devolve
    (df_anonimizado, {coluna: caminho_do_de_para})."""
    df_saida = df.copy()
    caminhos_de_para = {}

    for coluna, prefixo in colunas_anonimizar.items():
        if coluna not in df_saida.columns:
            print(f"aviso: coluna '{coluna}' nao existe na base, ignorada.")
            continue

        valores_unicos = df_saida[coluna].dropna().unique()
        mapeamento, caminho = carregar_ou_criar_mapeamento(
            coluna, valores_unicos, prefixo, seed, pasta_de_para, reutilizar)
        caminhos_de_para[coluna] = caminho

        mapeado = df_saida[coluna].astype(str).map(mapeamento)
        if prefixo is None:
            mapeado = mapeado.astype("Int64")
            if df_saida[coluna].dtype.kind in "iu":
                mapeado = mapeado.astype(df_saida[coluna].dtype)
        df_saida[coluna] = mapeado

    return df_saida, caminhos_de_para


def validar_anonimizacao(df_original, df_anonimizado, colunas_anonimizar):
    """Confere estrutura idêntica e ausência de vazamento dos valores
    originais nas colunas anonimizadas. Levanta AssertionError se algo falhar."""
    assert list(df_original.columns) == list(df_anonimizado.columns), \
        "colunas divergem entre original e anonimizado"
    assert df_original.shape == df_anonimizado.shape, \
        "numero de linhas/colunas divergiu"

    colunas_preservadas = [c for c in df_original.columns if c not in colunas_anonimizar]
    pd.testing.assert_frame_equal(df_original[colunas_preservadas],
                                   df_anonimizado[colunas_preservadas])

    for coluna in colunas_anonimizar:
        if coluna not in df_original.columns:
            continue
        originais = set(df_original[coluna].dropna().astype(str))
        anonimizados = set(df_anonimizado[coluna].dropna().astype(str))
        vazamento = originais & anonimizados
        assert not vazamento, f"valor original vazou em '{coluna}': {list(vazamento)[:5]}"
        assert df_original[coluna].nunique() == df_anonimizado[coluna].nunique(), \
            f"cardinalidade de '{coluna}' mudou apos a anonimizacao"

    print("validacao ok: estrutura identica, sem vazamento de valor original, "
          "cardinalidade preservada em", list(colunas_anonimizar.keys()))


## 4. Executar

In [ ]:
caminho_entrada = Path(CSV_ENTRADA)
df_original = pd.read_csv(caminho_entrada)
print(f"base carregada: {df_original.shape[0]:,} linhas, {df_original.shape[1]} colunas")
print(df_original.dtypes)

df_anonimizado, caminhos_de_para = aplicar_anonimizacao(
    df_original, COLUNAS_ANONIMIZAR, SEED, PASTA_DE_PARA_PRIVADA,
    REUTILIZAR_MAPEAMENTO_EXISTENTE)

pasta_saida = Path(PASTA_SAIDA_ANONIMIZADA)
pasta_saida.mkdir(parents=True, exist_ok=True)
caminho_saida = pasta_saida / f"{caminho_entrada.stem}_ANONIMIZADO.csv"
df_anonimizado.to_csv(caminho_saida, index=False)

print()
print("arquivo anonimizado salvo em:", caminho_saida)
for coluna, caminho in caminhos_de_para.items():
    print(f"de/para de '{coluna}' (PRIVADO, nao commitar/publicar):", caminho)


## 5. Validacao

In [ ]:
validar_anonimizacao(df_original, df_anonimizado, COLUNAS_ANONIMIZAR)

print()
print("amostra original:")
print(df_original.head(3).to_string())
print()
print("amostra anonimizada:")
print(df_anonimizado.head(3).to_string())
print()
print("SKUs unicos:", df_original["Prod_SKU"].nunique(),
      "->", df_anonimizado["Prod_SKU"].nunique())
print("Prod_Id unicos:", df_original["Prod_Id"].nunique(),
      "->", df_anonimizado["Prod_Id"].nunique())
print("Campanhas unicas:", df_original["Ped_Campanha"].nunique(),
      "->", df_anonimizado["Ped_Campanha"].nunique())
print("Categorias unicas:", df_original["Cat_CategoriaNome"].nunique(),
      "->", df_anonimizado["Cat_CategoriaNome"].nunique())
print("Cat_CategoriaId unicos:", df_original["Cat_CategoriaId"].nunique(),
      "->", df_anonimizado["Cat_CategoriaId"].nunique())


## Proximos passos

1. Confira o arquivo anonimizado (`*_ANONIMIZADO.csv`) manualmente antes de
   publica-lo.
2. Para rodar o pipeline principal (`TCC_Pipeline_Colab.ipynb`) sobre a base
   anonimizada, aponte o parametro `DADOS` dele para o caminho salvo em
   `caminho_saida`.
3. Nunca versione nem publique a pasta de `PASTA_DE_PARA_PRIVADA`.